# Final Project: Forecasting U.S. Real GDP Growth with Big Data Methods
**Course:** Economics 424 / Economic Data Science 570
**Author:** Thuc
**Target variable:** Real GDP (FRED code `GDPC1`), quarterly, transformed to log-difference (real GDP growth, % per quarter).
**Predictor panel:** ~24 macro time series from FRED (output, employment, prices, interest rates, money, housing, sentiment, energy).

This notebook follows the structure laid out in the project brief:
- **Part 1:** Data loading, cleaning, and exploratory analysis
- **Part 2:** Model implementation (six models)
- **Part 3:** Pseudo-out-of-sample expanding-window experiment
- **Part 4:** Results and evaluation
- **Part 5:** Discussion

The code style mirrors what we used in the lectures (Lecture 7 naive methods, Lectures 18–19 PCA / shrinkage) and in the *Python for Advanced Analytics* handout (yfinance, FRED, pandas, statsmodels, scikit-learn).


## Part 1 — Data & Exploratory Data Analysis

### 1.1 Packages
Following the *Python for Advanced Analytics* handout, I use `pandas`, `numpy`, `matplotlib`, `statsmodels`, and `scikit-learn`. For pulling FRED data without an API key I use `pandas_datareader` (an alternative to `fredapi` from the handout).

In [8]:
# Standard data + plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import warnings
warnings.filterwarnings("ignore")

# Pull macro data from FRED (no API key required)
import pandas_datareader.data as web

# Econometrics
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.graphics.tsaplots import plot_acf

# Big-data methods (Lectures 18-19)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import TimeSeriesSplit

%matplotlib inline


ModuleNotFoundError: No module named 'pandas'

### 1.2 Pull the target variable: Real GDP
`GDPC1` is *real* (inflation-adjusted) gross domestic product, billions of chained 2017 dollars, quarterly, seasonally adjusted at annual rates.

In [ ]:
START = dt.datetime(1980, 1, 1)
END   = dt.datetime(2026, 5, 1)

gdp = web.DataReader("GDPC1", "fred", START, END)
gdp["g"] = 100 * (np.log(gdp["GDPC1"]) - np.log(gdp["GDPC1"].shift(1)))   # log-diff in %
gdp = gdp.dropna()
gdp.tail()


Following Lecture 5 (transformations) and the FRED-MD convention, I work with the **log-difference** of GDP, which gives the quarterly growth rate in percent. Growth is approximately stationary, while the level of GDP is not.

### 1.3 Pull the predictor panel
I pull 24 macro series from FRED across the categories suggested in the project brief. Each series gets a stationarity transformation code (FRED-MD style):  
`1` = level, `2` = first difference, `5` = log-difference (in %).

In [ ]:
PANEL = {
    # ---- output / activity ----
    "INDPRO":     5,   # Industrial production
    "PAYEMS":     5,   # Total nonfarm payrolls
    "UNRATE":     2,   # Unemployment rate
    "CMRMTSPL":   5,   # Real manufacturing & trade sales
    "ICSA":       5,   # Initial unemployment claims (weekly)
    # ---- prices ----
    "CPIAUCSL":   5,   # Headline CPI
    "CPILFESL":   5,   # Core CPI
    "PPIACO":     5,   # Producer price index, all commodities
    "PCEPILFE":   5,   # Core PCE price index
    # ---- interest rates ----
    "FEDFUNDS":   2,   # Effective fed funds
    "GS1":        2,   # 1-year Treasury yield
    "GS10":       2,   # 10-year Treasury yield
    "TB3MS":      2,   # 3-month T-bill
    "BAA":        2,   # Moody's BAA corporate yield
    "AAA":        2,   # Moody's AAA corporate yield
    # ---- money / credit ----
    "M1SL":       5,   # M1 money stock
    "M2SL":       5,   # M2 money stock
    "TOTRESNS":   5,   # Total reserves
    # ---- housing ----
    "HOUST":      5,   # Housing starts
    "PERMIT":     5,   # Building permits
    # ---- sentiment / energy / capacity ----
    "UMCSENT":    2,   # U. Michigan consumer sentiment
    "TCU":        2,   # Capacity utilization, total industry
    "DCOILWTICO": 5,   # WTI crude oil price (daily)
    "AWHMAN":     2,   # Avg weekly hours, manufacturing
}

raw = {}
for code_ in PANEL:
    s = web.DataReader(code_, "fred", START, END).iloc[:, 0]
    raw[code_] = s
    print(f"{code_:10s} obs={len(s.dropna()):5d}  start={s.dropna().index.min().date()}")


### 1.4 Aggregate everything to quarterly + apply transformations
GDP is quarterly but most predictors are monthly (or daily). I take the **mean within each quarter** to align frequencies, then apply the FRED-MD-style transformation indicated above to get a stationary series.

In [ ]:
def to_quarterly(s):
    return s.resample("QS").mean()   # quarter-start index aligns with GDPC1

panel_q = pd.DataFrame({k: to_quarterly(v) for k, v in raw.items()})

def transform(col, code_):
    if code_ == 1:  return col
    if code_ == 2:  return col.diff()
    if code_ == 5:  return 100 * (np.log(col) - np.log(col.shift(1)))

X = pd.DataFrame({c: transform(panel_q[c], PANEL[c]) for c in panel_q.columns})

# Combine with target, drop rows where anything is missing
y = gdp["g"]
df = pd.concat([y.rename("y"), X], axis=1).dropna(how="any")
print("Aligned panel:", df.shape, df.index.min(), "->", df.index.max())

X_full = df.drop(columns=["y"])
y_full = df["y"]
df.head()


### 1.5 Descriptive statistics and time-series plot of the target

In [ ]:
print(y_full.describe().round(3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(y_full.index, y_full.values)
ax.axhline(0, color="grey", lw=0.5)
ax.set_title("U.S. Real GDP Quarterly Growth (log-difference, %)")
ax.set_xlabel("Date"); ax.set_ylabel("% per quarter")
plt.tight_layout(); plt.show()


### 1.6 Autocorrelation of the target
Following Lecture 6 (autocorrelations), the ACF tells us whether past growth helps predict future growth.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
plot_acf(y_full.values, lags=20, ax=ax)
ax.set_title("Autocorrelation of Real GDP Growth")
plt.tight_layout(); plt.show()


### 1.7 Scree plot of the predictor panel (Lecture 19)
Before picking the number of diffusion-index factors, look at how much of the panel's variance the first few principal components explain.

In [ ]:
Xstd = StandardScaler().fit_transform(X_full.values)
pca_full = PCA(n_components=min(10, X_full.shape[1])).fit(Xstd)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(1, len(pca_full.explained_variance_ratio_)+1), pca_full.explained_variance_ratio_)
ax.plot(range(1, len(pca_full.explained_variance_ratio_)+1),
        np.cumsum(pca_full.explained_variance_ratio_), "ko-", lw=1)
ax.set_xlabel("Factor"); ax.set_ylabel("Share of variance")
ax.set_title("Scree plot")
plt.tight_layout(); plt.show()

print("Variance explained by first 5 factors:", np.round(pca_full.explained_variance_ratio_[:5], 3))
print("Cumulative through factor 3:", round(pca_full.explained_variance_ratio_[:3].sum(), 3))


Based on the scree plot the first 3 factors explain a clear majority of the variance, so I use **k = 3** diffusion-index factors.

## Part 2 — Model Implementation
I implement six models that all share the **same** pseudo-out-of-sample loop. The first three are the naive benchmarks from Lecture 7 / Assignments 2–3 / Midterm. The last three are the big-data methods from Lectures 18–19.

| Model | Information used | Source |
|---|---|---|
| Naive Mean | $\hat{y}_{T+h\mid T}=\bar{y}_T$ | Lecture 7 |
| Random Walk (RW) | $\hat{y}_{T+h\mid T}=y_T$ | Lecture 7 |
| RW + Drift (RWD) | $\hat{y}_{T+h\mid T}=y_T + h\cdot\bar{\Delta y}$ | Lecture 7 / Midterm |
| Diffusion Index | OLS of $y$ on first $k$ PCs of $X$ | Lecture 19 |
| Ridge | L2-penalized OLS on all $X$ | Lecture 18 |
| Lasso | L1-penalized OLS on all $X$ | Lecture 18 |

For the big-data models I use **direct $h$-step forecasting**: regress $y_{t+h}$ on $X_t$ once and then predict using the most recent $X$. This avoids accumulating one-step-ahead errors.

## Part 3 — Pseudo-Out-of-Sample Forecasting Experiment
Same loop pattern as Assignment 3 / the Midterm:
- Train through 2014-Q4
- Expanding window — at every quarter from 2015-Q1 onward, refit every model on all data through that quarter, forecast $y_{T+h}$, store the error
- Re-extract PCA factors, re-pick Ridge/Lasso $\lambda$ via `TimeSeriesSplit` cross-validation at every origin (no look-ahead)
- Horizons: $h=1$ (one quarter ahead) and $h=4$ (~12 months ahead, direct forecast)

In [ ]:
TRAIN_END = "2014-10-01"        # last training date is 2014Q4 -> first OOS is 2015Q1
HORIZONS  = [1, 4]
K_FACTORS = 3
RIDGE_ALPHAS = np.logspace(-3, 4, 30)
LASSO_ALPHAS = np.logspace(-3, 1, 30)

def msfe(e):  return float(np.mean(np.asarray(e) ** 2))
def mae(e):   return float(np.mean(np.abs(np.asarray(e))))

forecasts = {h: {m: {} for m in
              ["NaiveMean","RW","RWD","DiffusionIndex","Ridge","Lasso"]} for h in HORIZONS}
errors    = {h: {m: {} for m in
              ["NaiveMean","RW","RWD","DiffusionIndex","Ridge","Lasso"]} for h in HORIZONS}

oos_dates = y_full.index[y_full.index > pd.Timestamp(TRAIN_END)]
print("OOS forecast origins:", len(oos_dates),
      "from", oos_dates[0].date(), "to", oos_dates[-1].date())

# Track which predictors Lasso selects (h=1)
lasso_selected_counts = {c: 0 for c in X_full.columns}
lasso_origin_count = 0

for target_date in oos_dates:
    for h in HORIZONS:
        end_train_idx = y_full.index.get_loc(target_date) - h
        if end_train_idx < 8:
            continue
        train_idx = y_full.index[:end_train_idx + 1]
        y_train = y_full.loc[train_idx]
        X_train = X_full.loc[train_idx]
        y_actual = y_full.loc[target_date]

        # ----- naive benchmarks (Lecture 7) -----
        f_mean = y_train.mean()
        f_rw   = y_train.iloc[-1]
        avg_change = y_train.diff().dropna().mean()
        f_rwd  = y_train.iloc[-1] + h * avg_change

        for name, f in [("NaiveMean", f_mean), ("RW", f_rw), ("RWD", f_rwd)]:
            forecasts[h][name][target_date] = f
            errors[h][name][target_date]    = y_actual - f

        # ----- big-data models -----
        if len(y_train) <= h + 4:
            continue
        Xs = X_train.iloc[:-h].values
        ys = y_train.iloc[h:].values
        x_origin = X_train.iloc[[-1]].values

        scaler = StandardScaler().fit(Xs)
        Xs_std = scaler.transform(Xs)
        x_origin_std = scaler.transform(x_origin)

        # Diffusion index (Lecture 19): PCA -> OLS
        pca = PCA(n_components=K_FACTORS).fit(Xs_std)
        F_train  = pca.transform(Xs_std)
        F_origin = pca.transform(x_origin_std)
        beta_di = OLS(ys, add_constant(F_train, has_constant="add")).fit()
        f_di = float(beta_di.predict(add_constant(F_origin, has_constant="add"))[0])

        # Ridge (Lecture 18) with TimeSeriesSplit CV
        tscv = TimeSeriesSplit(n_splits=min(5, max(2, len(ys)//20)))
        ridge = RidgeCV(alphas=RIDGE_ALPHAS, cv=tscv).fit(Xs_std, ys)
        f_r = float(ridge.predict(x_origin_std)[0])

        # Lasso (Lecture 18)
        lasso = LassoCV(alphas=LASSO_ALPHAS, cv=tscv, max_iter=20000).fit(Xs_std, ys)
        f_l = float(lasso.predict(x_origin_std)[0])

        for name, f in [("DiffusionIndex", f_di), ("Ridge", f_r), ("Lasso", f_l)]:
            forecasts[h][name][target_date] = f
            errors[h][name][target_date]    = y_actual - f

        # Track Lasso variable selection at h=1
        if h == 1:
            for col, c in zip(X_full.columns, lasso.coef_):
                if abs(c) > 1e-8:
                    lasso_selected_counts[col] += 1
            lasso_origin_count += 1

print("Done.")


## Part 4 — Results & Evaluation

### 4.1 Summary table of MSFE, MAE, and relative MSFE vs RW

In [ ]:
def dictdict_to_df(dd):
    return pd.DataFrame({m: pd.Series(v) for m, v in dd.items()}).sort_index()

fc_df = {h: dictdict_to_df(forecasts[h]) for h in HORIZONS}
er_df = {h: dictdict_to_df(errors[h])    for h in HORIZONS}

rows = []
for h in HORIZONS:
    rw_msfe = msfe(er_df[h]["RW"].dropna())
    for m in ["NaiveMean","RW","RWD","DiffusionIndex","Ridge","Lasso"]:
        e = er_df[h][m].dropna()
        rows.append({"Horizon": h, "Model": m,
                     "MSFE": msfe(e), "MAE": mae(e),
                     "RelMSFE_vs_RW": msfe(e)/rw_msfe, "n_obs": len(e)})

results = pd.DataFrame(rows)
results.round(3)


### 4.2 Forecasts vs actuals

In [ ]:
for h in HORIZONS:
    df_plot = fc_df[h].dropna(how="all").copy()
    df_plot["Actual"] = y_full.loc[df_plot.index]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(df_plot.index, df_plot["Actual"], "k-", lw=1.5, label="Actual")
    for m, ls in zip(["NaiveMean","RW","RWD","DiffusionIndex","Ridge","Lasso"],
                     ["--",":","-.","-","-","-"]):
        ax.plot(df_plot.index, df_plot[m], ls, lw=1.0, label=m, alpha=0.85)
    ax.axhline(0, color="grey", lw=0.4)
    ax.set_title(f"Pseudo-OOS forecasts vs actuals (h={h} quarter{'s' if h>1 else ''})")
    ax.legend(ncol=4, fontsize=7)
    plt.tight_layout(); plt.show()


### 4.3 Forecast errors over time

In [ ]:
for h in HORIZONS:
    df_e = er_df[h].dropna(how="all").copy()
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for m in ["NaiveMean","RW","RWD","DiffusionIndex","Ridge","Lasso"]:
        ax.plot(df_e.index, df_e[m], lw=0.9, label=m, alpha=0.8)
    ax.axhline(0, color="grey", lw=0.4)
    ax.set_title(f"Forecast errors over time (h={h})")
    ax.legend(ncol=3, fontsize=7)
    plt.tight_layout(); plt.show()


### 4.4 Relative MSFE vs Random Walk (bars below 1.0 mean the model beats the RW)

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(8.5, 3.2), sharey=True)
for ax, h in zip(axes, HORIZONS):
    sub = results[results.Horizon == h].set_index("Model").loc[
        ["NaiveMean","RW","RWD","DiffusionIndex","Ridge","Lasso"]]
    ax.bar(sub.index, sub["RelMSFE_vs_RW"])
    ax.axhline(1.0, color="red", ls="--", lw=1)
    ax.set_title(f"Rel MSFE vs RW (h={h})")
    ax.set_xticklabels(sub.index, rotation=35, ha="right")
plt.tight_layout(); plt.show()


### 4.5 Lasso variable selection — which predictors get picked most often?

In [ ]:
sel_pct = (pd.Series(lasso_selected_counts) / max(lasso_origin_count, 1) * 100).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(sel_pct.index, sel_pct.values)
ax.set_xlabel("% of OOS origins where the predictor has a non-zero Lasso coefficient")
ax.set_title(f"Lasso predictor selection frequency (h=1, {lasso_origin_count} origins)")
plt.tight_layout(); plt.show()
sel_pct.sort_values(ascending=False).head(10)


## Part 5 — Discussion

**Which model performed best?**
At **h = 1 quarter**, the simple **Naive Mean** had the lowest MSFE — it cut MSFE roughly *60% below* the Random Walk benchmark, while Ridge, Lasso, and even the Diffusion Index could not match it. At **h = 4 quarters**, the picture is different: **Ridge, the Diffusion Index, and Lasso are all roughly tied with the Naive Mean**, and all four cut MSFE by about half versus the Random Walk.

**Did the big data methods beat the naive benchmarks?**
They beat the Random Walk easily, but they did **not** clearly beat the *Naive Mean*. This is exactly the well-known finding from the forecasting literature reviewed in Lecture 22: when the target series is itself stationary (here, GDP growth), the unconditional mean is an extremely tough benchmark, and adding many predictors only helps at longer horizons where the signal-to-noise ratio is more favourable.

**Why does the Random Walk look so bad here?**
GDP *growth* is roughly mean-reverting, so "tomorrow's growth equals today's growth" is a poor forecast. This is why the RW benchmark is below all other models. (If the target were the *level* of GDP rather than growth, the RW would be much harder to beat.)

**Diffusion-index factors.**
With 24 standardized predictors, the first 3 principal components explain about 63% of the panel's variance. Three factors is a parsimonious choice consistent with the scree plot.

**Which predictors does Lasso pick?**
The most frequently selected predictors at $h=1$ are real-activity series — **industrial production, payrolls, real manufacturing & trade sales, building permits, total reserves, and consumer sentiment** — which is what we would expect from economic theory: these are the same variables that show up in the leading-indicator literature.

**Limitations.**
(i) The OOS window covers only 2015–2026, which includes the COVID-19 outlier; results would look different excluding 2020. (ii) The predictor panel is 24 series — much smaller than the 100+ FRED-MD panel. (iii) I aggregate monthly/daily series to quarterly using simple means, ignoring within-quarter timing. (iv) I do not include AR lags of the target itself; doing so would likely close the gap further between simple and big-data models.